<a href="https://colab.research.google.com/github/eslam158/player_detection/blob/main/player_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

MODEL_PATH = "/content/drive/MyDrive/best_football_yolo26.pt"

VIDEO_PATH = "/content/drive/MyDrive/LAUTARO RIMONTA IL NAPOLI E BUTTA GIÙ LO STADIO 🏟️ _ INTER 3-2 NAPOLI _ EXTENDED HIGHLIGHTS 📹🖤💙.mp4"

OUTPUT_PATH = "/content/drive/MyDrive/inter_napoli_WIDE_246_276_FINAL.mp4"

TRACKER_CONFIG = "/content/custom_botsort_reid.yaml"

START_TIME = 246
END_TIME = 276

print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)
print("Output:", OUTPUT_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model: /content/drive/MyDrive/best_football_yolo26.pt
Video: /content/drive/MyDrive/LAUTARO RIMONTA IL NAPOLI E BUTTA GIÙ LO STADIO 🏟️ _ INTER 3-2 NAPOLI _ EXTENDED HIGHLIGHTS 📹🖤💙.mp4
Output: /content/drive/MyDrive/inter_napoli_WIDE_246_276_FINAL.mp4


In [ ]:
%%writefile /content/custom_botsort_reid.yaml

tracker_type: botsort

track_high_thresh: 0.30
track_low_thresh: 0.08
new_track_thresh: 0.35

track_buffer: 120
match_thresh: 0.75
fuse_score: True

gmc_method: sparseOptFlow

proximity_thresh: 0.5
appearance_thresh: 0.25

with_reid: True
model: auto

Overwriting /content/custom_botsort_reid.yaml


In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque, defaultdict
from math import sqrt


# ============================================================
# 1. LOAD MODEL
# ============================================================

model = YOLO(MODEL_PATH)

print("Model loaded successfully")
print("Classes:", model.names)


# ============================================================
# 2. APPEARANCE FEATURE
# ============================================================

def appearance_feature(crop):
    """
    Extract a lightweight appearance feature.

    Uses:
    - HSV color histogram
    - Spatial HSV information
    """

    if crop is None or crop.size == 0:
        return None

    try:
        crop = cv2.resize(crop, (64, 128))

        hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)

        # ----------------------------------------------------
        # HSV histogram
        # ----------------------------------------------------

        hist_h = cv2.calcHist(
            [hsv],
            [0],
            None,
            [16],
            [0, 180]
        )

        hist_s = cv2.calcHist(
            [hsv],
            [1],
            None,
            [16],
            [0, 256]
        )

        hist_v = cv2.calcHist(
            [hsv],
            [2],
            None,
            [16],
            [0, 256]
        )

        hist_h = cv2.normalize(hist_h, hist_h).flatten()
        hist_s = cv2.normalize(hist_s, hist_s).flatten()
        hist_v = cv2.normalize(hist_v, hist_v).flatten()

        # ----------------------------------------------------
        # Spatial feature
        # ----------------------------------------------------

        small = cv2.resize(hsv, (8, 16))

        spatial = small.astype(np.float32).flatten()

        if np.linalg.norm(spatial) > 0:
            spatial = spatial / np.linalg.norm(spatial)

        # ----------------------------------------------------
        # Combine
        # ----------------------------------------------------

        feature = np.concatenate([
            hist_h,
            hist_s,
            hist_v,
            spatial
        ]).astype(np.float32)

        norm = np.linalg.norm(feature)

        if norm > 0:
            feature = feature / norm

        return feature

    except Exception:
        return None


# ============================================================
# 3. FEATURE SIMILARITY
# ============================================================

def cosine_similarity(a, b):

    if a is None or b is None:
        return 0.0

    denominator = (
        np.linalg.norm(a) *
        np.linalg.norm(b)
    )

    if denominator == 0:
        return 0.0

    value = np.dot(a, b) / denominator

    return float(np.clip(value, 0.0, 1.0))


# ============================================================
# 4. IOU
# ============================================================

def calculate_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection = (
        intersection_width *
        intersection_height
    )

    area1 = max(0, box1[2] - box1[0]) * max(
        0,
        box1[3] - box1[1]
    )

    area2 = max(0, box2[2] - box2[0]) * max(
        0,
        box2[3] - box2[1]
    )

    union = area1 + area2 - intersection

    if union <= 0:
        return 0.0

    return intersection / union


# ============================================================
# 5. CENTER
# ============================================================

def get_center(box):

    x1, y1, x2, y2 = box

    return (
        (x1 + x2) / 2,
        (y1 + y2) / 2
    )


# ============================================================
# 6. TRACK OBJECT
# ============================================================

class Track:

    def __init__(
        self,
        track_id,
        class_id,
        bbox,
        feature,
        frame_id
    ):

        self.track_id = track_id

        self.class_id = class_id

        self.bbox = bbox.copy()

        self.center = get_center(bbox)

        self.previous_center = self.center

        self.velocity = (0.0, 0.0)

        self.features = deque(
            maxlen=20
        )

        if feature is not None:
            self.features.append(feature)

        self.last_frame = frame_id

        self.missing = 0

        self.hits = 1

        self.total_visible_frames = 1

    # --------------------------------------------------------

    def update(
        self,
        bbox,
        feature,
        frame_id
    ):

        new_center = get_center(bbox)

        old_center = self.center

        self.previous_center = old_center

        self.velocity = (
            new_center[0] - old_center[0],
            new_center[1] - old_center[1]
        )

        self.center = new_center

        self.bbox = bbox.copy()

        if feature is not None:

            self.features.append(feature)

        self.last_frame = frame_id

        self.missing = 0

        self.hits += 1

        self.total_visible_frames += 1

    # --------------------------------------------------------

    def predict_center(self):

        return (
            self.center[0] + self.velocity[0],
            self.center[1] + self.velocity[1]
        )

    # --------------------------------------------------------

    def appearance_similarity(self, feature):

        if feature is None:
            return 0.0

        if len(self.features) == 0:
            return 0.0

        similarities = []

        for old_feature in self.features:

            sim = cosine_similarity(
                old_feature,
                feature
            )

            similarities.append(sim)

        if not similarities:
            return 0.0

        # Use the best few matches instead of only latest feature

        similarities.sort(reverse=True)

        top_values = similarities[:5]

        return float(np.mean(top_values))


# ============================================================
# 7. PERSISTENT ID MANAGER
# ============================================================

class PersistentIDManager:

    def __init__(self):

        self.tracks = {}

        self.next_id = 1

        # How long an old ID can remain alive
        self.max_missing = 180

        # Maximum distance in pixels
        self.max_distance = 180

        # Minimum appearance similarity
        self.appearance_threshold = 0.58

    # --------------------------------------------------------

    def create_track(
        self,
        class_id,
        bbox,
        feature,
        frame_id
    ):

        track_id = self.next_id

        self.next_id += 1

        self.tracks[track_id] = Track(
            track_id=track_id,
            class_id=class_id,
            bbox=bbox,
            feature=feature,
            frame_id=frame_id
        )

        return track_id

    # --------------------------------------------------------

    def calculate_score(
        self,
        track,
        bbox,
        feature
    ):

        # ----------------------------------------------------
        # Motion
        # ----------------------------------------------------

        predicted_center = track.predict_center()

        current_center = get_center(bbox)

        distance = sqrt(
            (predicted_center[0] - current_center[0]) ** 2 +
            (predicted_center[1] - current_center[1]) ** 2
        )

        # Normalize distance

        distance_score = max(
            0.0,
            1.0 - distance / self.max_distance
        )

        # ----------------------------------------------------
        # Appearance
        # ----------------------------------------------------

        appearance_score = (
            track.appearance_similarity(feature)
        )

        # ----------------------------------------------------
        # IoU
        # ----------------------------------------------------

        iou_score = calculate_iou(
            track.bbox,
            bbox
        )

        # ----------------------------------------------------
        # Final score
        # ----------------------------------------------------

        final_score = (
            0.60 * appearance_score +
            0.25 * distance_score +
            0.15 * iou_score
        )

        return (
            final_score,
            appearance_score,
            distance_score,
            iou_score,
            distance
        )

    # --------------------------------------------------------

    def match_detection(
        self,
        class_id,
        bbox,
        feature,
        frame_id
    ):

        candidates = []

        # ----------------------------------------------------
        # Search all old tracks of same class
        # ----------------------------------------------------

        for track_id, track in self.tracks.items():

            if track.class_id != class_id:
                continue

            if track.missing > self.max_missing:
                continue

            (
                score,
                appearance_score,
                distance_score,
                iou_score,
                distance
            ) = self.calculate_score(
                track,
                bbox,
                feature
            )

            # ------------------------------------------------
            # Candidate filtering
            # ------------------------------------------------

            if distance > self.max_distance:
                continue

            if (
                appearance_score <
                self.appearance_threshold
                and
                iou_score < 0.05
                and
                distance_score < 0.45
            ):
                continue

            candidates.append(
                (
                    score,
                    track_id,
                    appearance_score,
                    distance_score,
                    iou_score
                )
            )

        # ----------------------------------------------------
        # No candidate
        # ----------------------------------------------------

        if not candidates:

            return self.create_track(
                class_id,
                bbox,
                feature,
                frame_id
            )

        # ----------------------------------------------------
        # Best candidate
        # ----------------------------------------------------

        candidates.sort(
            key=lambda x: x[0],
            reverse=True
        )

        (
            best_score,
            best_id,
            best_appearance,
            best_distance,
            best_iou
        ) = candidates[0]

        # ----------------------------------------------------
        # Minimum matching score
        # ----------------------------------------------------

        if best_score >= 0.48:

            self.tracks[best_id].update(
                bbox,
                feature,
                frame_id
            )

            return best_id

        # ----------------------------------------------------
        # New identity
        # ----------------------------------------------------

        return self.create_track(
            class_id,
            bbox,
            feature,
            frame_id
        )

    # --------------------------------------------------------

    def update_missing(self, used_ids):

        for track_id, track in self.tracks.items():

            if track_id not in used_ids:

                track.missing += 1

    # --------------------------------------------------------

    def cleanup(self):

        delete_ids = []

        for track_id, track in self.tracks.items():

            if track.missing > self.max_missing:

                delete_ids.append(track_id)

        for track_id in delete_ids:

            del self.tracks[track_id]


# ============================================================
# 8. CLASS NAMES
# ============================================================

CLASS_NAMES = {
    0: "GOALKEEPER",
    1: "PLAYER",
    2: "REFEREE"
}


# ============================================================
# 9. VIDEO INFORMATION
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():

    raise RuntimeError(
        "Could not open video"
    )

fps = cap.get(cv2.CAP_PROP_FPS)

width = int(
    cap.get(cv2.CAP_PROP_FRAME_WIDTH)
)

height = int(
    cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
)

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

duration = total_frames / fps

print("=" * 70)
print("VIDEO INFORMATION")
print("=" * 70)
print("FPS:", fps)
print("Resolution:", width, "x", height)
print("Frames:", total_frames)
print("Duration:", round(duration, 2), "seconds")


# ============================================================
# 10. START / END FRAME
# ============================================================

start_frame = int(
    START_TIME * fps
)

end_frame = int(
    END_TIME * fps
)

cap.set(
    cv2.CAP_PROP_POS_FRAMES,
    start_frame
)

frames_to_process = (
    end_frame - start_frame
)

print()
print("Processing:")
print(
    f"{START_TIME}s -> {END_TIME}s"
)

print(
    "Frames:",
    frames_to_process
)


# ============================================================
# 11. OUTPUT VIDEO
# ============================================================

fourcc = cv2.VideoWriter_fourcc(
    *"mp4v"
)

out = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (width, height)
)

if not out.isOpened():

    raise RuntimeError(
        "Could not create output video"
    )


# ============================================================
# 12. ID MANAGER
# ============================================================

id_manager = PersistentIDManager()


# ============================================================
# 13. STATISTICS
# ============================================================

id_frames = defaultdict(int)

id_class = {}

frame_player_counts = []

total_player_detections = 0
total_referee_detections = 0
total_goalkeeper_detections = 0

frame_id = 0


# ============================================================
# 14. COLORS
# ============================================================

PLAYER_COLOR = (0, 255, 0)
REFEREE_COLOR = (0, 165, 255)
GOALKEEPER_COLOR = (255, 0, 0)


# ============================================================
# 15. PROCESS VIDEO
# ============================================================

while True:

    ret, frame = cap.read()

    if not ret:
        break

    if frame_id >= frames_to_process:
        break

    current_frame_number = (
        start_frame + frame_id
    )

    used_ids = set()

    current_player_count = 0

    # --------------------------------------------------------
    # YOLO TRACKING
    # --------------------------------------------------------

    results = model.track(
        frame,
        persist=True,
        tracker=TRACKER_CONFIG,
        conf=0.25,
        iou=0.5,
        verbose=False
    )

    if results is None or len(results) == 0:

        out.write(frame)

        frame_id += 1

        continue

    result = results[0]

    if result.boxes is None:

        out.write(frame)

        frame_id += 1

        continue

    boxes = result.boxes

    # --------------------------------------------------------
    # Process detections
    # --------------------------------------------------------

    for i in range(len(boxes)):

        cls = int(
            boxes.cls[i].item()
        )

        conf = float(
            boxes.conf[i].item()
        )

        # Only our 3 classes

        if cls not in CLASS_NAMES:
            continue

        x1, y1, x2, y2 = (
            boxes.xyxy[i]
            .cpu()
            .numpy()
            .astype(int)
        )

        # Safety

        x1 = max(0, min(width - 1, x1))
        y1 = max(0, min(height - 1, y1))

        x2 = max(0, min(width - 1, x2))
        y2 = max(0, min(height - 1, y2))

        if x2 <= x1 or y2 <= y1:
            continue

        bbox = np.array(
            [x1, y1, x2, y2],
            dtype=np.float32
        )

        # ----------------------------------------------------
        # Crop
        # ----------------------------------------------------

        crop = frame[
            y1:y2,
            x1:x2
        ]

        feature = appearance_feature(
            crop
        )

        # ----------------------------------------------------
        # Persistent ID
        # ----------------------------------------------------

        persistent_id = (
            id_manager.match_detection(
                class_id=cls,
                bbox=bbox,
                feature=feature,
                frame_id=current_frame_number
            )
        )

        used_ids.add(
            persistent_id
        )

        id_frames[
            persistent_id
        ] += 1

        id_class[
            persistent_id
        ] = cls

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        if cls == 1:

            total_player_detections += 1

            current_player_count += 1

        elif cls == 2:

            total_referee_detections += 1

        elif cls == 0:

            total_goalkeeper_detections += 1

        # ----------------------------------------------------
        # Draw
        # ----------------------------------------------------

        if cls == 1:

            color = PLAYER_COLOR

        elif cls == 2:

            color = REFEREE_COLOR

        else:

            color = GOALKEEPER_COLOR

        # Bounding box

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            color,
            2
        )

        label = (
            f"{CLASS_NAMES[cls]} "
            f"ID:{persistent_id} "
            f"{conf:.2f}"
        )

        # ----------------------------------------------------
        # Label background
        # ----------------------------------------------------

        (tw, th), baseline = (
            cv2.getTextSize(
                label,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                2
            )
        )

        label_y1 = max(
            0,
            y1 - th - baseline - 4
        )

        label_y2 = y1

        cv2.rectangle(
            frame,
            (x1, label_y1),
            (x1 + tw + 6, label_y2),
            color,
            -1
        )

        cv2.putText(
            frame,
            label,
            (x1 + 3, y1 - 5),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    # --------------------------------------------------------
    # Missing tracks
    # --------------------------------------------------------

    id_manager.update_missing(
        used_ids
    )

    id_manager.cleanup()

    # --------------------------------------------------------
    # Player count
    # --------------------------------------------------------

    frame_player_counts.append(
        current_player_count
    )

    # --------------------------------------------------------
    # Info on video
    # --------------------------------------------------------

    info = (
        f"Time: "
        f"{current_frame_number / fps:.1f}s"
        f"   Players: {current_player_count}"
    )

    cv2.putText(
        frame,
        info,
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    # --------------------------------------------------------
    # Write frame
    # --------------------------------------------------------

    out.write(frame)

    frame_id += 1

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if frame_id % 50 == 0:

        progress = (
            frame_id /
            frames_to_process *
            100
        )

        print(
            f"Progress: "
            f"{progress:.1f}% "
            f"({frame_id}/{frames_to_process})"
        )


# ============================================================
# 16. RELEASE
# ============================================================

cap.release()
out.release()



print()
print("=" * 70)
print("VIDEO PROCESSING FINISHED")
print("=" * 70)

Model loaded successfully
Classes: {0: 'goal keeper', 1: 'player', 2: 'refree'}
VIDEO INFORMATION
FPS: 25.0
Resolution: 1920 x 1080
Frames: 13063
Duration: 522.52 seconds

Processing:
246s -> 276s
Frames: 750
Progress: 6.7% (50/750)
Progress: 13.3% (100/750)
Progress: 20.0% (150/750)
Progress: 26.7% (200/750)
Progress: 33.3% (250/750)
Progress: 40.0% (300/750)
Progress: 46.7% (350/750)
Progress: 53.3% (400/750)
Progress: 60.0% (450/750)
Progress: 66.7% (500/750)
Progress: 73.3% (550/750)
Progress: 80.0% (600/750)
Progress: 86.7% (650/750)
Progress: 93.3% (700/750)
Progress: 100.0% (750/750)

VIDEO PROCESSING FINISHED


In [ ]:
# ============================================================
# FINAL REPORT
# ============================================================

print()
print("=" * 70)
print("FINAL PERSISTENT RE-ID REPORT")
print("=" * 70)

print(
    f"Time: {START_TIME}s -> {END_TIME}s"
)

print(
    f"Frames: {frames_to_process}"
)

print()


# ============================================================
# IDs BY CLASS
# ============================================================

player_ids = sorted([
    track_id
    for track_id, cls in id_class.items()
    if cls == 1
])

referee_ids = sorted([
    track_id
    for track_id, cls in id_class.items()
    if cls == 2
])

goalkeeper_ids = sorted([
    track_id
    for track_id, cls in id_class.items()
    if cls == 0
])


print("PLAYER")
print(
    "Unique Player IDs:",
    len(player_ids)
)

print()

print("REFEREE")
print(
    "Unique Referee IDs:",
    len(referee_ids)
)

print()

print("GOALKEEPER")
print(
    "Unique Goalkeeper IDs:",
    len(goalkeeper_ids)
)


# ============================================================
# PLAYER DURATIONS
# ============================================================

print()
print("PLAYER ID DURATIONS")

player_durations = []

for track_id in player_ids:

    frames = id_frames[track_id]

    seconds = frames / fps

    player_durations.append(
        (
            track_id,
            frames,
            seconds
        )
    )

player_durations.sort(
    key=lambda x: x[2],
    reverse=True
)

for track_id, frames, seconds in player_durations:

    if seconds >= 3:

        print(
            f"ID {track_id}: "
            f"{frames} frames | "
            f"{seconds:.2f}s"
        )


# ============================================================
# REFEREE DURATIONS
# ============================================================

print()
print("REFEREE ID DURATIONS")

referee_durations = []

for track_id in referee_ids:

    frames = id_frames[track_id]

    seconds = frames / fps

    referee_durations.append(
        (
            track_id,
            frames,
            seconds
        )
    )

referee_durations.sort(
    key=lambda x: x[2],
    reverse=True
)

for track_id, frames, seconds in referee_durations:

    print(
        f"ID {track_id}: "
        f"{frames} frames | "
        f"{seconds:.2f}s"
    )


# ============================================================
# GOALKEEPER DURATIONS
# ============================================================

print()
print("GOALKEEPER ID DURATIONS")

goalkeeper_durations = []

for track_id in goalkeeper_ids:

    frames = id_frames[track_id]

    seconds = frames / fps

    goalkeeper_durations.append(
        (
            track_id,
            frames,
            seconds
        )
    )

goalkeeper_durations.sort(
    key=lambda x: x[2],
    reverse=True
)

for track_id, frames, seconds in goalkeeper_durations:

    print(
        f"ID {track_id}: "
        f"{frames} frames | "
        f"{seconds:.2f}s"
    )


# ============================================================
# PLAYER COUNT
# ============================================================

print()
print("PLAYER COUNT")

if frame_player_counts:

    average_players = np.mean(
        frame_player_counts
    )

    maximum_players = np.max(
        frame_player_counts
    )

    minimum_players = np.min(
        frame_player_counts
    )

else:

    average_players = 0
    maximum_players = 0
    minimum_players = 0


print(
    f"Average: {average_players:.2f}"
)

print(
    f"Maximum: {maximum_players}"
)

print(
    f"Minimum: {minimum_players}"
)


# ============================================================
# ID STABILITY
# ============================================================

print()
print("ID STABILITY")

if player_durations:

    average_visibility = np.mean([
        x[2]
        for x in player_durations
    ])

else:

    average_visibility = 0


print(
    f"Average player visibility: "
    f"{average_visibility:.2f} seconds"
)


# ============================================================
# STABILITY BREAKDOWN
# ============================================================

stable_5 = sum(
    1
    for _, _, seconds in player_durations
    if seconds >= 5
)

stable_10 = sum(
    1
    for _, _, seconds in player_durations
    if seconds >= 10
)

stable_15 = sum(
    1
    for _, _, seconds in player_durations
    if seconds >= 15
)

stable_20 = sum(
    1
    for _, _, seconds in player_durations
    if seconds >= 20
)


print()
print("STABILITY BREAKDOWN")

print(
    f"IDs >= 5 sec: {stable_5}"
)

print(
    f"IDs >= 10 sec: {stable_10}"
)

print(
    f"IDs >= 15 sec: {stable_15}"
)

print(
    f"IDs >= 20 sec: {stable_20}"
)


# ============================================================
# DETECTION TOTALS
# ============================================================

print()
print("DETECTION TOTALS")

print(
    "Player detections:",
    total_player_detections
)

print(
    "Referee detections:",
    total_referee_detections
)

print(
    "Goalkeeper detections:",
    total_goalkeeper_detections
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("OUTPUT")
print(OUTPUT_PATH)

print()
print("DONE.")


FINAL PERSISTENT RE-ID REPORT
Time: 246s -> 276s
Frames: 750

PLAYER
Unique Player IDs: 55

REFEREE
Unique Referee IDs: 8

GOALKEEPER
Unique Goalkeeper IDs: 15

PLAYER ID DURATIONS
ID 34: 666 frames | 26.64s
ID 1: 577 frames | 23.08s
ID 40: 532 frames | 21.28s
ID 45: 515 frames | 20.60s
ID 62: 502 frames | 20.08s
ID 64: 443 frames | 17.72s
ID 55: 438 frames | 17.52s
ID 59: 437 frames | 17.48s
ID 63: 423 frames | 16.92s
ID 58: 414 frames | 16.56s
ID 60: 371 frames | 14.84s
ID 56: 364 frames | 14.56s
ID 57: 339 frames | 13.56s
ID 54: 308 frames | 12.32s
ID 69: 265 frames | 10.60s
ID 46: 222 frames | 8.88s
ID 70: 220 frames | 8.80s
ID 53: 206 frames | 8.24s
ID 13: 163 frames | 6.52s
ID 65: 108 frames | 4.32s
ID 44: 97 frames | 3.88s
ID 7: 93 frames | 3.72s
ID 15: 88 frames | 3.52s
ID 61: 88 frames | 3.52s
ID 4: 87 frames | 3.48s
ID 41: 75 frames | 3.00s

REFEREE ID DURATIONS
ID 52: 317 frames | 12.68s
ID 33: 50 frames | 2.00s
ID 78: 49 frames | 1.96s
ID 49: 38 frames | 1.52s
ID 10: 36 fr

In [ ]:
from google.colab import files

files.download(OUTPUT_PATH)